# Projeto #1: Churn Prediction

## Projeto #1: Churn Prediction

**Domínio:** SaaS/Streaming/Telecom — qualquer empresa com clientes recorrentes
**Pergunta:** Qual cliente vai cancelar? Por quê? Como reter?
**Conceitos cobertos:** Structured Outputs (W2), RAG tradicional + adaptativo (W3-4),
Tool Use (W5), Agentic Loop com stopping rule (W5-6), LangGraph linear (W6-7),
Confidence Scoring (W7), Observability (W8)

Todo o código usa **mocks** — não precisa de API key pra rodar. Troque
`mock_llm_call` por uma chamada real à Anthropic API quando for pra produção.

In [ ]:
!pip install -q langgraph pydantic

import random
from typing import Literal, Optional
from pydantic import BaseModel, Field

### 1. Schema estruturado (Week 2)

Structured outputs garantem parsing confiável — não dependemos do LLM
"decidir" o formato da resposta.

In [ ]:
class ChurnPrediction(BaseModel):
    customer_id: str
    will_churn: bool
    confidence: float = Field(ge=0.0, le=1.0)
    risk_factors: list[str]
    recommended_action: str

class CustomerState(BaseModel):
    customer_id: str
    tenure_months: int
    monthly_spend: float
    support_tickets_90d: int
    engagement_score: float  # 0-1
    similar_churned: list[dict] = []
    prediction: Optional[ChurnPrediction] = None
    strategy_attempts: int = 0
    verified: bool = False

### 2. Dados sintéticos (substituem o CSV de 10k clientes)

Gerador determinístico — dá pra trocar por `datasets/customers.csv` real depois.

In [ ]:
def generate_customers(n: int = 20, seed: int = 42) -> list[CustomerState]:
    random.seed(seed)
    customers = []
    for i in range(n):
        customers.append(CustomerState(
            customer_id=f"cust_{i:04d}",
            tenure_months=random.randint(1, 48),
            monthly_spend=round(random.uniform(20, 500), 2),
            support_tickets_90d=random.randint(0, 8),
            engagement_score=round(random.uniform(0.1, 1.0), 2),
        ))
    return customers

customers = generate_customers()
print(f"✓ {len(customers)} clientes sintéticos gerados")
print(customers[0])

### 3. RAG leve — clientes similares que já cancelaram (Week 3-4)

RAG "tradicional": busca por similaridade em vez de embeddings de verdade
(substitua por um vector DB real — Pinecone/Chroma — em produção).

In [ ]:
def retrieve_similar_churned(customer: CustomerState, churn_history: list[dict]) -> list[dict]:
    """Adaptive RAG: só retrieva se o cliente tiver sinais de risco.
    Evita custo de retrieval quando não precisa (10x mais eficiente)."""
    needs_retrieval = customer.support_tickets_90d >= 3 or customer.engagement_score < 0.4
    if not needs_retrieval:
        return []

    def distance(c):
        return abs(c["tenure_months"] - customer.tenure_months) + \
               abs(c["engagement_score"] - customer.engagement_score) * 10
    return sorted(churn_history, key=distance)[:3]

CHURN_HISTORY = [
    {"customer_id": "hist_01", "tenure_months": 3, "engagement_score": 0.2, "reason": "onboarding ruim"},
    {"customer_id": "hist_02", "tenure_months": 14, "engagement_score": 0.3, "reason": "concorrente mais barato"},
    {"customer_id": "hist_03", "tenure_months": 2, "engagement_score": 0.15, "reason": "não usou o produto"},
]

### 4. Tool: consultar histórico de suporte (Week 5)

In [ ]:
def tool_query_support_history(customer_id: str) -> dict:
    """Tool que o agente chama pra buscar contexto adicional."""
    random.seed(hash(customer_id) % 1000)
    return {
        "customer_id": customer_id,
        "avg_resolution_hours": round(random.uniform(1, 48), 1),
        "satisfaction_score": round(random.uniform(1, 5), 1),
    }

### 5. "LLM" mockado + loop com stopping rule (Week 5-6-7)

In [ ]:
def mock_llm_call(customer: CustomerState, strategy: str) -> ChurnPrediction:
    """MOCK — troque por client.messages.create(...) da Anthropic.
    3 estratégias diferentes simulam prompts distintos (few-shot, CoT, com RAG)."""
    base_risk = (
        (1 - customer.engagement_score) * 0.5
        + min(customer.support_tickets_90d / 8, 1) * 0.3
        + (1 if customer.tenure_months < 3 else 0) * 0.2
    )
    strategy_boost = {"heuristic": 0.0, "cot": 0.08, "rag_augmented": 0.15}[strategy]
    confidence = min(0.95, base_risk * 0.6 + 0.3 + strategy_boost)

    risk_factors = []
    if customer.engagement_score < 0.4:
        risk_factors.append("baixo engajamento")
    if customer.support_tickets_90d >= 3:
        risk_factors.append("muitos tickets de suporte")
    if customer.tenure_months < 3:
        risk_factors.append("cliente novo (onboarding crítico)")
    if customer.similar_churned:
        risk_factors.append(f"{len(customer.similar_churned)} clientes similares já cancelaram")

    return ChurnPrediction(
        customer_id=customer.customer_id,
        will_churn=base_risk > 0.5,
        confidence=round(confidence, 2),
        risk_factors=risk_factors or ["sem sinais fortes de risco"],
        recommended_action="oferecer desconto + onboarding assistido" if base_risk > 0.5 else "monitorar",
    )

def churn_agentic_loop(customer: CustomerState, min_confidence: float = 0.85, max_iterations: int = 3) -> CustomerState:
    """Loop Engineering: decide -> act -> observe -> repete até confiança
    suficiente ou esgotar o orçamento de iterações (stopping rule)."""
    strategies = ["heuristic", "cot", "rag_augmented"]
    for i in range(max_iterations):
        customer.strategy_attempts = i + 1
        strategy = strategies[min(i, len(strategies) - 1)]
        prediction = mock_llm_call(customer, strategy)
        customer.prediction = prediction
        print(f"  [{customer.customer_id}] tentativa {i+1} ({strategy}): confidence={prediction.confidence}")
        if prediction.confidence >= min_confidence:
            break
    return customer

### 6. Grafo (Week 6-7): Gather → Retrieve → Predict → Verify → Recommend

In [ ]:
from langgraph.graph import StateGraph, START, END

def node_gather(state: CustomerState) -> CustomerState:
    return state

def node_retrieve(state: CustomerState) -> CustomerState:
    state.similar_churned = retrieve_similar_churned(state, CHURN_HISTORY)
    return state

def node_predict(state: CustomerState) -> CustomerState:
    return churn_agentic_loop(state)

def node_verify(state: CustomerState) -> CustomerState:
    """Harness Verifier: escalona pra humano se confiança ficar baixa mesmo
    após esgotar as estratégias."""
    if state.prediction and state.prediction.confidence < 0.60:
        state.prediction.recommended_action = "ESCALAR PARA HUMANO: confiança baixa"
    state.verified = True
    return state

graph = StateGraph(CustomerState)
graph.add_node("gather", node_gather)
graph.add_node("retrieve", node_retrieve)
graph.add_node("predict", node_predict)
graph.add_node("verify", node_verify)
graph.add_edge(START, "gather")
graph.add_edge("gather", "retrieve")
graph.add_edge("retrieve", "predict")
graph.add_edge("predict", "verify")
graph.add_edge("verify", END)

churn_agent = graph.compile()

### 7. Rodando ponta a ponta

Nota: `churn_agent.invoke(c)` retorna um **novo** state — não muta `c` original.
Guardamos os resultados numa lista pra usar depois (armadilha comum com LangGraph).

In [ ]:
results = []
for c in customers[:5]:
    result = churn_agent.invoke(c)
    p = result["prediction"] if isinstance(result, dict) else result.prediction
    print(f"→ {c.customer_id}: churn={p.will_churn} conf={p.confidence} ação='{p.recommended_action}'\n")
    results.append(result)

### 8. Observability mínima (Week 8)

In [ ]:
def log_prediction(result):
    """Structured log — em produção isso vai pro Cloud Logging (GCP)."""
    get = (lambda k: result[k]) if isinstance(result, dict) else (lambda k: getattr(result, k))
    p = get("prediction")
    print({
        "event": "churn_prediction",
        "customer_id": get("customer_id"),
        "will_churn": p.will_churn,
        "confidence": p.confidence,
        "iterations": get("strategy_attempts"),
        "escalated": "ESCALAR" in p.recommended_action,
    })

for r in results[:3]:
    log_prediction(r)

**Próximos passos pra produção:**
- Trocar `mock_llm_call` por `anthropic.Anthropic().messages.create(...)`
- Trocar `retrieve_similar_churned` por um vector DB real (Chroma/Pinecone)
- Persistir `CustomerState` no Firestore
- Deploy no Cloud Run (ver `docs/source-material/plano_estudos_ai_agents_gcp.md`)
- Avaliação real: comparar `will_churn` previsto vs churn observado após 30 dias